# USGS Streamflow Download — Iowa

Downloads daily mean discharge (ft³/s, parameter code 00060) for all active
Iowa stream gauges from USGS NWIS for the same date range as the EPA water
quality data (2015-01-01 – 2025-12-31).

**Outputs**
- `data/tabular/streamflow/raw/usgs-iowa-gauges.csv` — site metadata (lat/lon, name)
- `data/tabular/streamflow/raw/usgs-iowa-discharge.csv` — daily discharge per site

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import dataretrieval.nwis as nwis
from pathlib import Path

RAW_DIR = Path('../../data/tabular/streamflow/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = '2015-01-01'
END_DATE   = '2025-12-31'

## 1. Site metadata

In [2]:
sites, _ = nwis.get_info(stateCd='IA', parameterCd='00060', siteType='ST')

keep_cols = ['site_no', 'station_nm', 'dec_lat_va', 'dec_long_va',
             'drain_area_va', 'huc_cd', 'county_cd']
sites = sites[[c for c in keep_cols if c in sites.columns]].copy()
sites.rename(columns={
    'station_nm':   'station_name',
    'dec_lat_va':   'latitude',
    'dec_long_va':  'longitude',
    'drain_area_va': 'drain_area_sqmi',
    'huc_cd':       'huc8',
    'county_cd':    'county_fips'
}, inplace=True)

out_sites = RAW_DIR / 'usgs-iowa-gauges.csv'
sites.to_csv(out_sites, index=False)
print(f'Saved {len(sites)} gauges → {out_sites}')
sites.head()

Saved 703 gauges → ../../data/tabular/streamflow/raw/usgs-iowa-gauges.csv


,site_no,station_name,latitude,longitude,drain_area_sqmi,huc8,county_fips
0,05317650,"Blue Earth River near Lakota, IA",43.429679,-94.069958,64.6,7020009,109
1,05387300,"Upper Iowa River at Chester, IA",43.491077,-92.362116,141.0,7060002,89
2,05387400,"Upper Iowa River near Kendallville, IA",43.464611,-92.038889,273.0,7060002,191
3,05387440,"Upper Iowa River at Bluffton, IA",43.406913,-91.899046,367.0,7060002,191
4,05387500,"Upper Iowa River at Decorah, IA",43.304889,-91.795543,511.0,7060002,191


## 2. Daily discharge

Downloads all Iowa stream sites in a single API call. This may take a few
minutes. The result is a multi-index DataFrame (site_no, datetime); we reset
the index before saving.

In [3]:
print('Downloading daily discharge for all Iowa gauges...')
discharge, _ = nwis.get_dv(
    stateCd='IA',
    parameterCd='00060',
    start=START_DATE,
    end=END_DATE,
)
print(f'Downloaded {len(discharge):,} site-day rows')

Downloaded 560,896 site-day rows


In [4]:
discharge = discharge.reset_index()

# Rename the NWIS column to something readable
discharge.rename(columns={
    'site_no':               'site_no',
    'datetime':              'date',
    '00060_Mean':            'discharge_cfs',
    '00060_Mean_cd':         'discharge_cd',   # data-quality code
}, inplace=True)

# Keep only the columns we actually renamed (others may not exist)
wanted = ['site_no', 'date', 'discharge_cfs', 'discharge_cd']
discharge = discharge[[c for c in wanted if c in discharge.columns]]

out_discharge = RAW_DIR / 'usgs-iowa-discharge.csv'
discharge.to_csv(out_discharge, index=False)
print(f'Saved {len(discharge):,} rows → {out_discharge}')
discharge.head()

Saved 560,896 rows → ../../data/tabular/streamflow/raw/usgs-iowa-discharge.csv


,site_no,date,discharge_cfs,discharge_cd
0,05387440,2015-01-01 00:00:00+00:00,127.0,"A, e"
1,05387440,2015-01-02 00:00:00+00:00,169.0,"A, e"
2,05387440,2015-01-03 00:00:00+00:00,173.0,"A, e"
3,05387440,2015-01-04 00:00:00+00:00,154.0,"A, e"
4,05387440,2015-01-05 00:00:00+00:00,124.0,"A, e"


## 3. Quick summary

In [5]:
print('Gauges:         ', len(sites))
print('Site-day rows:  ', f'{len(discharge):,}')
print('Date range:     ', discharge['date'].min(), '–', discharge['date'].max())
print('Missing values: ', discharge['discharge_cfs'].isna().sum(), 'of', len(discharge))

Gauges:          703
Site-day rows:   560,896
Date range:      2015-01-01 00:00:00+00:00 – 2025-12-31 00:00:00+00:00
Missing values:  4020 of 560896
